In [0]:
import uuid
from datetime import datetime, timezone
 
 
 
class SilverConfig:
   
    ADLS_ACCOUNT_NAME : str = ""
 
    # Container roots (populated by init)
    LANDING_ROOT  : str = ""
    BRONZE_ROOT   : str = ""
    SILVER_ROOT   : str = ""
    LOGGING_ROOT  : str = ""
 
    # Run-level metadata (set once per notebook execution)
    RUN_ID    : str = ""
    RUN_TS    : datetime = None
    DATE_PATH : str = ""   # YYYY/MM/DD  — used in log file paths
 
    @classmethod
    def init(cls, adls_account_name: str) -> None:
        
        cls.ADLS_ACCOUNT_NAME = adls_account_name
 
        cls.LANDING_ROOT = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net"
        cls.BRONZE_ROOT  = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/bronze"
        cls.SILVER_ROOT  = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/silver"
        cls.LOGGING_ROOT = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/logs"
 
    
        cls.RUN_ID    = str(uuid.uuid4())
        cls.RUN_TS    = datetime.now(timezone.utc)
        cls.DATE_PATH = cls.RUN_TS.strftime("%Y/%m/%d")
 

In [0]:

 
class BronzePaths:
    
    COINS_MARKETS : str = ""
    OHLC          : str = ""
    MARKET_CHART  : str = ""
    TRENDING      : str = ""
    GLOBAL        : str = ""
 
    @classmethod
    def init(cls, bronze_root: str) -> None:
        cls.COINS_MARKETS = f"{bronze_root}/coins_markets_raw"
        cls.OHLC          = f"{bronze_root}/ohlc_raw"
        cls.MARKET_CHART  = f"{bronze_root}/market_chart_raw"
        cls.TRENDING      = f"{bronze_root}/trending_raw"
        cls.GLOBAL        = f"{bronze_root}/global_raw"
 
 

In [0]:

 
class SilverPaths:
    
    MARKET_SNAPSHOT    : str = ""
    OHLC_HISTORY       : str = ""
    HOURLY_TIMESERIES  : str = ""
    TRENDING_COINS     : str = ""
    GLOBAL_STATS       : str = ""
 
    @classmethod
    def init(cls, silver_root: str) -> None:
        cls.MARKET_SNAPSHOT   = f"{silver_root}/market_snapshot"
        cls.OHLC_HISTORY      = f"{silver_root}/ohlc_history"
        cls.HOURLY_TIMESERIES = f"{silver_root}/hourly_timeseries"
        cls.TRENDING_COINS    = f"{silver_root}/trending_coins"
        cls.GLOBAL_STATS      = f"{silver_root}/global_stats"
 
 

In [0]:

# WATERMARK TABLE PATH
#
# WHY A WATERMARK TABLE?
#   Instead of scanning ALL Bronze rows with MAX(bronze_loaded_at) every run
#   (which grows more expensive as Bronze accumulates), we maintain a tiny
#   Delta table with one row per Bronze source table, recording the last
#   bronze_loaded_at timestamp that Silver successfully processed.

class WatermarkPaths:
    
    WATERMARK_TABLE : str = ""
 
    
    COINS_MARKETS  = "coins_markets_raw"
    OHLC           = "ohlc_raw"
    MARKET_CHART   = "market_chart_raw"
    TRENDING       = "trending_raw"
    GLOBAL         = "global_raw"
 
    @classmethod
    def init(cls, silver_root: str) -> None:
        
        cls.WATERMARK_TABLE = f"{silver_root}/_watermarks/bronze_watermarks"
 


In [0]:

 
class LogPaths:
   
    MARKET_SNAPSHOT   : str = ""
    OHLC_HISTORY      : str = ""
    HOURLY_TIMESERIES : str = ""
    TRENDING_COINS    : str = ""
    GLOBAL_STATS      : str = ""
 
    @classmethod
    def init(cls, logging_root: str, date_path: str, run_id: str) -> None:
        
        base = f"{logging_root}/silver"
        cls.MARKET_SNAPSHOT   = f"{base}/market_snapshot/{date_path}/run_{run_id}.json"
        cls.OHLC_HISTORY      = f"{base}/ohlc_history/{date_path}/run_{run_id}.json"
        cls.HOURLY_TIMESERIES = f"{base}/hourly_timeseries/{date_path}/run_{run_id}.json"
        cls.TRENDING_COINS    = f"{base}/trending_coins/{date_path}/run_{run_id}.json"
        cls.GLOBAL_STATS      = f"{base}/global_stats/{date_path}/run_{run_id}.json"
 

In [0]:

# WHY MERGE INSTEAD OF OVERWRITE?
#   If the pipeline re-runs (due to a bug fix or retry), MERGE ensures we don't
#   create duplicate rows. A row already in Silver with the same key is left
#   unchanged; only genuinely new rows are inserted.

 
class MergeKeys:
    """
    Primary key column combinations for each Silver Delta table.
    These are used in the MERGE condition:
        "existing.coin_id = new.coin_id AND existing.snapshot_date = new.snapshot_date"
    """
 
    # market_snapshot: one row per coin per day
    MARKET_SNAPSHOT = ["coin_id", "snapshot_date"]
 
    # ohlc_history: one row per OHLC candle per coin per timestamp
    OHLC_HISTORY    = ["coin_id", "ohlc_timestamp"]
 
    # hourly_timeseries: one row per coin per hour
    HOURLY_TIMESERIES = ["coin_id", "hour_timestamp"]
 
    # trending_coins: one row per trending position per day
    TRENDING_COINS  = ["trend_run_date", "coin_id"]
 
    # global_stats: one row per day (global has no coin_id)
    GLOBAL_STATS    = ["stats_date"]

In [0]:

 
class DataQuality:
    
    MIN_PRICE_USD          = 0.0        # current_price_usd must be > this
    MIN_MARKET_CAP_USD     = 0          # market_cap_usd must be > this (0 = allow any positive)
    MAX_MARKET_CAP_RANK    = 10_000     # sanity check: no coin should rank > 10,000
    MAX_PRICE_CHANGE_PCT   = 10_000.0   # anything over 10,000% change is likely bad data
 
 
    REQUIRE_HIGH_GTE_LOW   = True
    # close must be >= 0 (no negative prices)
    MIN_OHLC_PRICE         = 0.0
 
    # hourly_timeseries
    MIN_VOLUME_USD         = 0          # volume cannot be negative
 
    # global_stats
    MIN_ACTIVE_CRYPTOS     = 1          # must track at least 1 active crypto
    MIN_BTC_DOMINANCE_PCT  = 0.0        # BTC dominance as % (0–100)
    MAX_BTC_DOMINANCE_PCT  = 100.0
 
    MAX_DROP_FRACTION      = 0.40       # 40% — more than this is suspicious
 

In [0]:

 
class SilverColumns:
    
 
    MARKET_SNAPSHOT = [
        "coin_id",
        "symbol",
        "name",
        "snapshot_date",
        "snapshot_timestamp",
        "current_price_usd",
        "market_cap_usd",
        "market_cap_rank",
        "total_volume_24h_usd",
        "high_24h_usd",
        "low_24h_usd",
        "price_change_24h_usd",
        "price_change_pct_24h",
        "circulating_supply",
        "total_supply",
        "ath_usd",
        "ath_date",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    OHLC_HISTORY = [
        "coin_id",
        "ohlc_timestamp",
        "ohlc_date",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    HOURLY_TIMESERIES = [
        "coin_id",
        "hour_timestamp",
        "price_usd",
        "market_cap_usd",
        "volume_usd",
        "silver_processed_timestamp",
    ]
 
    TRENDING_COINS = [
        "trend_run_date",
        "trend_position",
        "coin_id",
        "coin_name",
        "coin_symbol",
        "market_cap_rank",
        "coin_price",
        "price_change_24h_percent",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    GLOBAL_STATS = [
        "stats_date",
        "stats_timestamp",
        "total_market_cap_usd",
        "total_volume_24h_usd",
        "btc_dominance_pct",
        "eth_dominance_pct",
        "active_cryptos",
        "market_cap_change_pct_24h",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 

In [0]:

 
class SecretConfig:
    SCOPE             = "key-vault-scope"
    KEY_ADLS_NAME     = "adls-key"
 
 
def init_silver_config(adls_account_name: str) -> None:
    
    SilverConfig.init(adls_account_name)
    BronzePaths.init(SilverConfig.BRONZE_ROOT)
    SilverPaths.init(SilverConfig.SILVER_ROOT)
    WatermarkPaths.init(SilverConfig.SILVER_ROOT)
    LogPaths.init(
        SilverConfig.LOGGING_ROOT,
        SilverConfig.DATE_PATH,
        SilverConfig.RUN_ID
    )
 